# What makes the toxic reviews toxic? Evidence phrases and types

For every review that is still toxic after the cheating re-check (1,669), Gemini points at up to 3 **exact words or phrases** that make it toxic and gives each one a **type**. Then we count.

**Types** follow common toxicity schemes: Perspective API attributes (insult, profanity, identity attack, threat, sexually explicit), Risch and Krestel (2020: profanity, insults, threats, identity hate, otherwise toxic), and the game-toxicity literature (griefing, trolling, toxic community; Beres et al. 2021; Fair Play Alliance and ADL *Disruption and Harms in Online Gaming Framework*, 2020). For identity attacks Gemini also says which group (ethnicity or nationality, gender, and so on). Each piece of evidence also has a **target**: teammate, opponent, players in general, a nationality, the developer.

**Safety check.** A quote must appear word for word in the review. `quote_in_review` is `False` when it does not, so invented quotes are caught and can be dropped.

**Caveats.** The evidence is Gemini's own explanation, not proof of why it decided. Counts describe Gemini's labels, not player behavior. A phrase that points at a nationality is a statement in the review, not a fact about that nationality.

Reads `data/gemini_harm_recheck_*.csv` (read only) and writes NEW files: `data/gemini_toxic_evidence_*.jsonl` and `.csv`.

**Run All twice:** first with `SAMPLE = 50`, then `SAMPLE = None` for all.

## 1. Setup

In [ ]:
import os, sys, json, importlib
import pandas as pd
from dotenv import load_dotenv
from google import genai

sys.path.insert(0, "../scripts")
import toxic_evidence
importlib.reload(toxic_evidence)

load_dotenv("../.env")
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 2. Choose the reviews (the one thing to change)

In [ ]:
SAMPLE = 50   # <-- test first with 50 toxic reviews. Then set to None and Run All again.

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0
MODEL_TAG = GEMINI_MODEL.replace("/", "_")

SOURCE = f"../data/gemini_harm_recheck_{MODEL_TAG}.csv"     # read only
CACHE  = f"../data/gemini_toxic_evidence_{MODEL_TAG}.jsonl" # NEW
OUT    = f"../data/gemini_toxic_evidence_{MODEL_TAG}.csv"   # NEW

rc = pd.read_csv(SOURCE)
toxic = rc[rc["harm_mention_v2"] == "yes"].sample(frac=1, random_state=0).reset_index(drop=True)  # fixed shuffle
df = toxic.head(SAMPLE) if SAMPLE else toxic
print(f"{len(toxic):,} toxic reviews in total, this run covers {len(df):,}")
df["game"].value_counts()

## 3. Run

Every review is saved the moment it finishes. A second run skips what is saved.

In [ ]:
with open("../prompts/toxic_evidence.txt", "r") as f:
    SYSTEM_EV = f.read()

run = toxic_evidence.EvidenceRun(df, client, GEMINI_MODEL, SYSTEM_EV, CACHE, temperature=TEMPERATURE, workers=8)
run.start()

## 4. Save the evidence table (new file) and check the quotes

In [ ]:
ev = run.export(OUT)

ok_rate = ev["quote_in_review"].mean()
print(f"\nQuotes found word for word in the review: {ev['quote_in_review'].sum():,} of {len(ev):,} ({ok_rate:.1%})")
print("The rest are kept in the file but left out of the counts below.")
ev = ev[ev["quote_in_review"]].copy()
ev.head(10)[["game", "quote", "meaning_en", "type", "identity_group", "target"]]

## 5. Which types of toxicity?

Share of toxic reviews that have **at least one** piece of evidence of each type, per game. A review can have several types, so the columns do not add to 100%.

In [ ]:
n_rev = df.groupby("game")["key"].nunique()
has = ev.drop_duplicates(["key", "type"]).groupby(["game", "type"]).size().unstack(fill_value=0)
share = (has.div(n_rev, axis=0) * 100).round(1)
share.loc["All three"] = (ev.drop_duplicates(["key", "type"]).groupby("type").size() / df["key"].nunique() * 100).round(1)
share = share.reindex(columns=toxic_evidence.TYPES, fill_value=0)
print("Toxic reviews per game:", n_rev.to_dict())
share

## 6. What are the most common phrases?

The top meanings (in English) for each type, so phrases in different languages can be counted together.

In [ ]:
pd.set_option("display.max_colwidth", 80)
for t in toxic_evidence.TYPES:
    s = ev[ev["type"] == t]
    if len(s) == 0:
        continue
    print(f"\n=== {t}: {len(s):,} pieces of evidence in {s['key'].nunique():,} reviews ===")
    print(s["meaning_en"].str.lower().value_counts().head(8).to_string())

## 7. Who is it aimed at, and which groups?

In [ ]:
print("Target of the toxic evidence (share of all evidence)")
display((ev["target"].value_counts(normalize=True) * 100).round(1).to_frame("%"))

ia = ev[ev["type"] == "identity_attack"]
print(f"\nIdentity attacks: {ia['key'].nunique():,} reviews. Which group?")
display((ia.drop_duplicates(["key", "identity_group"])["identity_group"].value_counts()).to_frame("reviews"))
print("Identity attacks by game (reviews)")
display(ia.drop_duplicates("key").groupby("game").size().to_frame("reviews"))

## 8. Human spot check

Gemini explains its own labels, so check a few by hand. This writes 30 random pieces of evidence to a new CSV. Fill `human_ok` with `yes` or `no` (is the type right and does the quote really show toxicity?). Then the share of `yes` tells us how far to trust the counts above.

In [ ]:
check = ev.sample(min(30, len(ev)), random_state=7)[["key", "game", "quote", "meaning_en", "type", "identity_group", "target", "review"]].copy()
check["human_ok"] = ""
CHECK_OUT = f"../data/toxic_evidence_spotcheck_{MODEL_TAG}.csv"
check.to_csv(CHECK_OUT, index=False)
print(f"Saved {len(check)} rows -> {CHECK_OUT}")
check[["quote", "meaning_en", "type", "target"]].head(10)